# SRG-Tracker — Interactive notebook product

Paste or edit one student's ordered weekly course history and press **Run prediction**. This version intentionally uses a notebook form; a user-friendly frontend is planned for V3.0. Predictions come from synthetic data and must never be used for automatic academic decisions.

In [ ]:
from pathlib import Path
import json
import sys

COLAB_ROOT = Path('/content/SRG-Tracker')
PROJECT_ROOT = COLAB_ROOT if (COLAB_ROOT / 'src').is_dir() else (Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent)
SRC_DIR = PROJECT_ROOT / 'src'
MODELS_DIR = PROJECT_ROOT / 'models'
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f'Cannot locate src directory from {Path.cwd()}')
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

try:
    import ipywidgets as widgets
    from IPython.display import display
    from src.inference import predict_history
    from src.preprocessing import history_from_group, load_split
except ImportError as error:
    raise ImportError('Install project dependencies with: pip install -r requirements.txt') from error

In [ ]:
# Load a safe synthetic example. Replace its values with your own hypothetical data.
validation = load_split('validation')
example_attempt = next(iter(validation.groupby('attempt_id')))[1]
example_history = history_from_group(example_attempt, cutoff=7)
history_input = widgets.Textarea(
    value=json.dumps(example_history, indent=2),
    description='History:',
    layout=widgets.Layout(width='100%', height='420px'),
    style={'description_width': '80px'},
)
run_button = widgets.Button(description='Run prediction', button_style='primary')
output = widgets.Output()
warning = widgets.HTML('<b>Warning:</b> synthetic-data demonstration only; human review is required.')

In [ ]:
def on_run_prediction(_button):
    with output:
        output.clear_output()
        try:
            history = json.loads(history_input.value)
            if not isinstance(history, list):
                raise ValueError('History JSON must be a list of weekly records.')
            prediction = predict_history(history)
        except json.JSONDecodeError as error:
            print(f'Invalid JSON near line {error.lineno}, column {error.colno}: {error.msg}')
            return
        except (ValueError, FileNotFoundError) as error:
            print(f'Cannot produce prediction: {error}')
            return
        print(json.dumps(prediction, indent=2))

run_button.on_click(on_run_prediction)
display(warning, history_input, run_button, output)